# Neural Language Models & Seq2Seq — Hands-On Notebook

Companion to: [Neural Language Models](../docs/01_foundations/neural_language_models.md), [Sequence-to-Sequence](../docs/01_foundations/sequence_to_sequence.md)

**What you'll build:**
- A character-level LSTM language model from scratch
- Gate value visualization for LSTMs
- Teacher forcing demonstration
- Text generation with HuggingFace GPT-2
- A translation demo with MarianMT

## Section 1: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Section 2: From Counting to Learning

N-gram models count word co-occurrences. Neural language models **LEARN** patterns. Instead of storing a probability table, a neural LM uses a network to compute P(next_word | previous_words). (In this notebook we apply the same idea at the **character** level: P(next_char | previous_chars).)

## Section 3: Character-Level LSTM Language Model

Let's build a model that predicts the next character. We'll train on a small text and see it learn to generate English-like text.

In [ ]:
TRAIN_TEXT = """
The quick brown fox jumps over the lazy dog. Learning language from characters is a small step
toward understanding how neural networks predict what comes next. Patterns in spelling and
common words emerge from training on text. We use a recurrent network to remember context
from earlier characters in the sequence. Each prediction depends on the hidden state built
from all previous inputs. Over many epochs the model reduces its loss and produces more
coherent continuations. This toy example fits in memory and runs quickly on a laptop.
Students can experiment with sequence length, hidden size, and learning rate to see how
training dynamics change. Character models are simple but illustrate the same ideas used
in large word-based and subword language models at scale.
""".replace("\n", " ").strip()

print("Training text length (chars):", len(TRAIN_TEXT))
print(TRAIN_TEXT[:200], "...")

In [ ]:
chars = sorted(set(TRAIN_TEXT))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

print("Vocabulary size:", vocab_size)
print("Sample mapping:", list(char2idx.items())[:8])

In [ ]:
SEQ_LEN = 48
encoded = [char2idx[c] for c in TRAIN_TEXT]

inputs_list = []
targets_list = []
for i in range(0, len(encoded) - SEQ_LEN):
    inputs_list.append(encoded[i : i + SEQ_LEN])
    targets_list.append(encoded[i + 1 : i + SEQ_LEN + 1])

train_inputs = torch.tensor(inputs_list, dtype=torch.long)
train_targets = torch.tensor(targets_list, dtype=torch.long)

print("Input batch shape:", train_inputs.shape)
print("Example input (indices):", train_inputs[0][:12].tolist())
print("Example target (next chars):", "".join(idx2char[int(i)] for i in train_targets[0][:12]))

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        logits = self.fc(out)
        return logits, hidden


embed_dim, hidden_dim = 64, 128
lm = CharLSTM(vocab_size, embed_dim, hidden_dim).to(device)
opt = optim.Adam(lm.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

n_epochs = 50
bi = train_inputs.to(device)
bt = train_targets.to(device)

for epoch in range(1, n_epochs + 1):
    lm.train()
    opt.zero_grad()
    logits, _ = lm(bi)
    loss = criterion(logits.reshape(-1, vocab_size), bt.reshape(-1))
    loss.backward()
    opt.step()
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | loss = {loss.item():.4f}")

print("Final training loss:", loss.item())

## Section 4: Visualizing LSTM Gates

What's happening inside the LSTM? Let's look at gate activations for a sample input.

PyTorch's built-in `nn.LSTM` uses a fused implementation that does not expose individual gates. Below we use an **explicit LSTM cell** (the same update equations) so we can read forget, input, and output gates at each timestep—this is what you would hook into for debugging in a custom implementation.

In [ ]:
class ExplicitLSTMCell(nn.Module):
    """One LSTM cell with exposed sigmoid/tanh gates (batch, feat)."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ih = nn.Linear(input_size, 4 * hidden_size)
        self.W_hh = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x, hc):
        h, c = hc
        gates = self.W_ih(x) + self.W_hh(h)
        i_gate, f_gate, g_tanh, o_gate = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i_gate)
        f = torch.sigmoid(f_gate)
        g = torch.tanh(g_tanh)
        o = torch.sigmoid(o_gate)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new, {"input": i, "forget": f, "output": o, "cell": c_new}


class CharLSTMExplicit(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.cell = ExplicitLSTMCell(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim

    def forward(self, x, return_gates=False):
        # x: (batch, seq)
        b, t = x.shape
        e = self.embed(x)
        h = torch.zeros(b, self.hidden_dim, device=x.device, dtype=e.dtype)
        c = torch.zeros(b, self.hidden_dim, device=x.device, dtype=e.dtype)
        logits_steps = []
        gate_hist = {"input": [], "forget": [], "output": [], "cell": []}
        for ti in range(t):
            xt = e[:, ti, :]
            h, c, gates = self.cell(xt, (h, c))
            logits_steps.append(self.fc(h))
            if return_gates:
                for k in gate_hist:
                    gate_hist[k].append(gates[k].detach().cpu())
        logits = torch.stack(logits_steps, dim=1)
        if return_gates:
            for k in gate_hist:
                gate_hist[k] = torch.stack(gate_hist[k], dim=1)  # (batch, time, hidden)
            return logits, gate_hist
        return logits


lm_ex = CharLSTMExplicit(vocab_size, embed_dim, hidden_dim).to(device)
opt_ex = optim.Adam(lm_ex.parameters(), lr=0.005)
for epoch in range(1, 31):
    opt_ex.zero_grad()
    logits, _ = lm_ex(bi)
    loss_ex = criterion(logits.reshape(-1, vocab_size), bt.reshape(-1))
    loss_ex.backward()
    opt_ex.step()
print("Explicit model quick-train loss:", loss_ex.item())

sample = "The quick brown"
sample_idx = torch.tensor([[char2idx[c] for c in sample]], device=device)
lm_ex.eval()
with torch.no_grad():
    _, gh = lm_ex(sample_idx, return_gates=True)

# Mean over hidden units -> one value per gate per timestep
forget_v = gh["forget"][0].mean(dim=-1).numpy()
input_v = gh["input"][0].mean(dim=-1).numpy()
output_v = gh["output"][0].mean(dim=-1).numpy()
cell_v = gh["cell"][0].mean(dim=-1).numpy()

mat = np.vstack([forget_v, input_v, output_v, cell_v])
fig, ax = plt.subplots(figsize=(max(8, len(sample) * 0.35), 3))
im = ax.imshow(mat, aspect="auto", cmap="viridis")
ax.set_yticks(range(4))
ax.set_yticklabels(["forget", "input", "output", "cell (mean)"])
ax.set_xticks(range(len(sample)))
ax.set_xticklabels(list(sample))
ax.set_xlabel("Character position (timestep)")
ax.set_title("LSTM gate & cell signals (mean over hidden units)")
fig.colorbar(im, ax=ax, label="Activation")
plt.tight_layout()
plt.show()

print("Gate tensor shapes (batch, time, hidden):", gh["forget"].shape)

Notice how the forget gate learns when to drop old information and the input gate learns when to write new information.

## Section 5: Generating Text (temperature)

We sample from the softmax using temperature $T$: logits / T. Lower $T$ sharpens the distribution (more conservative); higher $T$ flattens it (more random).

In [ ]:
def sample_next(logits, temperature):
    logits = logits / max(temperature, 1e-6)
    p = F.softmax(logits, dim=-1)
    return torch.multinomial(p, 1).item()


@torch.no_grad()
def generate_text(model, seed, n_chars, temperature, device=device):
    model.eval()
    idxs = [char2idx[c] for c in seed]
    for _ in range(n_chars):
        x = torch.tensor([idxs[-SEQ_LEN:]], dtype=torch.long, device=device)
        logits, _ = model(x)
        next_i = sample_next(logits[0, -1], temperature)
        idxs.append(next_i)
    return "".join(idx2char[i] for i in idxs)


seed_str = "We use a "
for T in [0.5, 1.0, 1.5]:
    out = generate_text(lm, seed_str, n_chars=200, temperature=T)
    print(f"\n--- temperature = {T} (first 200 generated chars after seed) ---")
    print(out[len(seed_str) : len(seed_str) + 200])

Low temperature → safer, more repetitive. High temperature → more creative, more chaotic.

## Section 6: Teacher Forcing

During training, do we feed the model its own (possibly wrong) predictions, or the correct answer? **Teacher forcing** means we always feed the ground-truth previous token. That is like learning to cook with a recipe versus improvising from a mistaken step.

We train two identical models: one with teacher forcing (standard next-token prediction on aligned inputs) and one **without** teacher forcing where each step conditions on the model's previous greedy prediction. The no–teacher-forcing update uses **batch size 1** (one sliding window) so the recurrent state matches a single autoregressive pass; comparing loss scales to full-batch teacher forcing is illustrative, not apples-to-apples on compute.

In [ ]:
def train_teacher_forcing(model, opt, epochs=40):
    losses = []
    model.train()
    for epoch in range(1, epochs + 1):
        opt.zero_grad()
        logits, _ = model(bi)
        loss = criterion(logits.reshape(-1, vocab_size), bt.reshape(-1))
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


def train_no_teacher_forcing(model, opt, epochs=40):
    """Train on one sequence (batch=1): each step feeds the model its previous greedy prediction."""
    losses = []
    model.train()
    xs = bi[:1]
    b, t = xs.shape
    for epoch in range(1, epochs + 1):
        opt.zero_grad()
        inp = xs[:, :1]
        total_loss = 0.0
        for step in range(t - 1):
            logits, _ = model(inp)
            pred_logits = logits[:, -1, :]
            target = xs[:, step + 1]
            step_loss = criterion(pred_logits, target)
            total_loss = total_loss + step_loss
            pred_char = pred_logits.argmax(dim=-1, keepdim=True)
            inp = torch.cat([inp, pred_char], dim=1)
        loss = total_loss / (t - 1)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


torch.manual_seed(123)
m_tf = CharLSTM(vocab_size, embed_dim, hidden_dim).to(device)
o_tf = optim.Adam(m_tf.parameters(), lr=0.005)
loss_tf = train_teacher_forcing(m_tf, o_tf, epochs=40)

torch.manual_seed(123)
m_ntf = CharLSTM(vocab_size, embed_dim, hidden_dim).to(device)
o_ntf = optim.Adam(m_ntf.parameters(), lr=0.005)
loss_ntf = train_no_teacher_forcing(m_ntf, o_ntf, epochs=40)

plt.figure(figsize=(8, 4))
plt.plot(loss_tf, label="Teacher forcing", linewidth=2)
plt.plot(loss_ntf, label="No teacher forcing (greedy feedback)", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss: teacher forcing vs. sequential greedy feedback")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final loss (teacher forcing):", loss_tf[-1])
print("Final loss (no teacher forcing):", loss_ntf[-1])

Teacher forcing speeds up training but can cause **exposure bias** — the model rarely sees its own mistakes at training time, so it may struggle to recover from errors during generation.

## Section 7: Text Generation with GPT-2

Our tiny character LSTM implements the same *idea* as GPT-2: predict the next token from context — just at a vastly smaller scale. Here we load a real pretrained model.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_gpt2.eval()
prompt = "Neural language models learn"
inputs = tokenizer.encode(prompt, return_tensors="pt")
max_new = 80

with torch.no_grad():
    out_greedy = model_gpt2.generate(
        inputs,
        max_new_tokens=max_new,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    out_topk = model_gpt2.generate(
        inputs,
        max_new_tokens=max_new,
        do_sample=True,
        top_k=50,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    out_topp = model_gpt2.generate(
        inputs,
        max_new_tokens=max_new,
        do_sample=True,
        top_p=0.9,
        top_k=0,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Greedy decoding (num_beams=1, do_sample=False)")
print(tokenizer.decode(out_greedy[0], skip_special_tokens=True))
print("\nTop-k sampling (do_sample=True, top_k=50)")
print(tokenizer.decode(out_topk[0], skip_special_tokens=True))
print("\nNucleus / top-p sampling (do_sample=True, top_p=0.9)")
print(tokenizer.decode(out_topp[0], skip_special_tokens=True))

Different decoding strategies can produce very different continuations from the **same** underlying model.

## Section 8: Translation with MarianMT

Sequence-to-sequence models power machine translation: an encoder reads the source sentence; a decoder generates the target sentence token by token.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-fr"
tok_mt = MarianTokenizer.from_pretrained(model_name)
model_mt = MarianMTModel.from_pretrained(model_name)
model_mt.eval()

sentences_en = [
    "Hello, how are you today?",
    "Machine translation maps one language to another.",
    "The weather is nice this afternoon.",
    "Students learn faster with clear examples.",
    "Neural networks can capture long-range dependencies.",
]

print("Tokenize → encode (IDs) → model.generate → decode → string\n")
for sent in sentences_en:
    batch = tok_mt([sent], return_tensors="pt", padding=True, truncation=True)
    print("EN:", sent)
    print("  token IDs (first 12):", batch["input_ids"][0][:12].tolist())
    with torch.no_grad():
        gen = model_mt.generate(**batch, max_length=128, num_beams=4)
    fr = tok_mt.decode(gen[0], skip_special_tokens=True)
    print("FR:", fr)
    print()

## Section 9: Visualizing Hidden States (GPT-2)

We feed a sentence through GPT-2 with `output_hidden_states=True` and compare each layer's vector for a token to the **final** layer's vector at the same position using cosine similarity.

In [ ]:
sentence_hs = "The scientist published surprising results."
enc_hs = tokenizer(sentence_hs, return_tensors="pt")
with torch.no_grad():
    out_hs = model_gpt2(**enc_hs, output_hidden_states=True)

hidden_states = out_hs.hidden_states  # tuple: embedding output + each transformer layer
n_layers = len(hidden_states)
seq_len = enc_hs["input_ids"].shape[1]
final_h = hidden_states[-1][0]  # (seq, hidden)

sim_matrix = np.zeros((n_layers, seq_len))
for li, h in enumerate(hidden_states):
    h = h[0]  # (seq, hidden)
    for t in range(seq_len):
        a = F.normalize(h[t], dim=-1)
        b = F.normalize(final_h[t], dim=-1)
        sim_matrix[li, t] = (a * b).sum().item()

tokens = tokenizer.convert_ids_to_tokens(enc_hs["input_ids"][0])
fig, ax = plt.subplots(figsize=(max(10, seq_len * 0.45), 5))
im = ax.imshow(sim_matrix, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xlabel("Token position")
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_ylabel("Layer (0 = embed, higher = deeper)")
ax.set_yticks(range(0, n_layers, max(1, n_layers // 12)))
ax.set_title("Cosine similarity to final-layer hidden state (same token position)")
fig.colorbar(im, ax=ax, label="Cosine similarity")
plt.tight_layout()
plt.show()

print("Number of hidden state tensors (incl. embedding stem):", n_layers)

Earlier layers often track more local and syntactic structure; deeper layers integrate context—one reason **parameter-efficient fine-tuning** can focus on the last layers only.

## Section 10: Key Takeaways

- **Neural LMs** learn a conditional distribution over the next token instead of storing huge count tables.
- **LSTMs** control memory with gates; visualizing gates builds intuition (in practice, use an explicit cell or custom module if you need the values).
- **Temperature** and **decoding strategy** (greedy, top-k, top-p) strongly affect open-ended generation.
- **Teacher forcing** trains quickly but differs from free-running generation—**exposure bias** is the mismatch.
- **Seq2seq** (e.g., MarianMT) maps an input sequence to an output sequence; encoder–decoder attention (in full architectures) connects them.
- **Layer-wise hidden states** in transformers show progressive abstraction; similarity to the last layer rises in deeper layers for aligned positions.

---

*Run all cells top to bottom. Sections 7–9 require `transformers` and will download model weights on first use.*